In [ ]:
"""
Script: train_spectral_classifier.ipynb

Description:
    Loads spectral moment CSV data for various knot types, constructs a PyTorch Dataset and DataLoader,
    defines a 2D convolutional classifier (Classifier2D), trains it on spectral inputs, and saves the model.

Sections:
    1. Imports & Device Setup
    2. Configuration: hyperparameters & data classes
    3. Data Loading: spectral CSV to feature tensors
    4. Dataset & DataLoader Preparation
    5. Model Definition: Classifier2D architecture
    6. Model Initialization & Summary
    7. Training Utilities: loops & plotting
    8. Training Loop
    9. Model Saving
"""

In [ ]:
# -----------------------------------------------------------------------------
# 1. Imports & Device Setup
# -----------------------------------------------------------------------------
import sys
sys.path.append('../')  # Add project root to PYTHONPATH
import time
import json, csv
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm import trange
import matplotlib.pyplot as plt
from torchsummary import summary

from extra_functions_package.all_knots_functions import *  # Knot utilities

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
# -----------------------------------------------------------------------------
# 2. Configuration: hyperparameters & data classes
# -----------------------------------------------------------------------------
hyperparams = {
    'learning_rate': 2e-5,  # Initial learning rate
    'patience': 0,          # LR scheduler patience epochs
    'decay_epoch': 25,      # Epoch to trigger decay
    'factor': 0.2,          # LR decay factor
    'batch_size': 64        # Mini-batch size
}
num_epochs = 50           # Total training epochs
print_every = 1           # Epoch interval for logging

# Define knot classes for spectral data labels
knot_types = {
    'standard_14': 0, 'standard_16': 1, 'standard_18': 2,
    '30both': 3, '30oneZ': 4, 'optimized': 5, 'pm_03_z': 6,
    '30oneX': 7, '15oneZ': 8,
    'trefoil_standard_12': 9, 'trefoil_optimized': 10
}
knots = list(knot_types.keys())
num_classes = len(knots)
folders = [
    '../HOPFS_L270_0.05_1000_64x64x64_v1',
    '../HOPFS_L270_0.15_1000_64x64x64_v1',
    '../HOPFS_L270_0.25_1000_64x64x64_v1'
]


In [ ]:
# -----------------------------------------------------------------------------
# 3. Data Loading: spectral CSV to feature tensors
# -----------------------------------------------------------------------------
X_features = []  # List of feature vectors
Y_labels = []    # Corresponding one-hot labels
global_count = 0
csv.field_size_limit(10_000_000)  # Handle large JSON entries

for folder in folders:
    for knot in knots:
        path = f"{folder}/data_{knot}_spectr.csv"
        try:
            with open(path, 'r') as f:
                reader = csv.reader(f)
                for row in reader:
                    # Parse JSON list: [l1, l2, p1, p2, idx, complex moments...]
                    data = json.loads(row[0])
                    l1, l2, p1, p2 = data[:4]
                    raw = data[5:]
                    # Convert to complex moments and reshape
                    moments = np.array([c[0] + 1j*c[1] for c in raw])
                    moments = moments.reshape((l2-l1+1, p2-p1+1))
                    # Normalize magnitude spectrum
                    moments /= np.linalg.norm(moments)
                    # Use absolute values as real feature inputs
                    X_features.append(np.abs(moments).ravel())
                    Y_labels.append(knot_types[knot])
                    global_count += 1
        except FileNotFoundError:
            print(f"Missing file: {path}")
        except json.JSONDecodeError:
            print(f"Invalid JSON: {path}")

print(f"Loaded {global_count} spectral samples ({global_count//num_classes} per class)")


In [ ]:
# -----------------------------------------------------------------------------
# 4. Dataset & DataLoader Preparation
# -----------------------------------------------------------------------------
X_np = np.stack(X_features)              # Shape: [N, features]
y_np = np.array(Y_labels)               # Shape: [N]
X_tensor = torch.tensor(X_np).float()   # Float tensor for inputs
y_tensor = F.one_hot(torch.tensor(y_np), num_classes).float()  # One-hot labels

dataset = TensorDataset(X_tensor, y_tensor)
train_loader = DataLoader(dataset, batch_size=hyperparams['batch_size'], shuffle=True)

In [ ]:
# -----------------------------------------------------------------------------
# 5. Model Definition: Classifier2D architecture
# -----------------------------------------------------------------------------
# Helper to build convolutional stages
def conv_stage_2d(configs):
    layers = []
    for in_ch, out_ch, k, s, p in configs:
        layers += [nn.Conv2d(in_ch, out_ch, k, s, p), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True)]
    return nn.Sequential(*layers)

# Helper to build pooling layers
def create_pool2d(cfg):
    return None if cfg is None else nn.MaxPool2d(*cfg)

class Classifier2D(nn.Module):
    """2D CNN for spectral features"""
    def __init__(self, stages, pooling, num_classes, height, width):
        super().__init__()
        self.height, self.width = height, width
        self.features = nn.Sequential()
        # Build conv + pooling
        for i, st in enumerate(stages):
            self.features.add_module(f"conv{i}", conv_stage_2d(st))
            pool = create_pool2d(pooling[i] if i < len(pooling) else None)
            if pool: self.features.add_module(f"pool{i}", pool)
        # Determine flattened size
        dummy = torch.rand(1, 1, height, width)
        flat_dim = int(self.features(dummy).reshape(1, -1).size(1))
        # Fully connected head
        self.fc1 = nn.Linear(flat_dim, 256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x):
        # Reshape flat input to image
        batch = x.size(0)
        x = x.view(batch, 1, self.height, self.width)
        x = self.features(x)
        x = x.view(batch, -1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

# Example conv/pool config
stages_cfg = [
    [(1, 32, 3, 1, 1), (32, 32, 3, 1, 1)],
    [(32, 64, 3, 1, 1), (64, 64, 3, 1, 1)]
]
pool_cfg = [(2,2,1), (2,2,1)]
# Compute input image dimensions from raw feature length
feat_len = X_tensor.shape[1]
# In this script, height & width known from spectral grid
height, width = data[1]-data[0]+1, data[3]-data[2]+1

In [ ]:
# -----------------------------------------------------------------------------
# 6. Model Initialization & Summary
# -----------------------------------------------------------------------------
model = Classifier2D(stages_cfg, pool_cfg, num_classes, height, width).to(device)
print(model)
summary(model, input_size=(1, height, width))

In [ ]:
# -----------------------------------------------------------------------------
# 7. Training Utilities: loops & plotting
# -----------------------------------------------------------------------------
criterion = nn.CrossEntropyLoss().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=hyperparams['learning_rate'])
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=hyperparams['factor'],
                              patience=hyperparams['patience'], verbose=True)

def loop_train():
    model.train(); total=0.0
    for Xb, yb in train_loader:
        Xb,yb=Xb.to(device),yb.to(device)
        optimizer.zero_grad()
        out=model(Xb)
        loss=criterion(out, torch.argmax(yb,1))
        loss.backward(); optimizer.step()
        total+=loss.item()
    return total/len(train_loader)

def plot_losses(losses):
    plt.plot(losses); plt.title('Loss'); plt.show()

In [ ]:
# -----------------------------------------------------------------------------
# 8. Training Loop
# -----------------------------------------------------------------------------
losses=[]
start=time.time()
for ep in trange(num_epochs):
    l=loop_train(); losses.append(l)
    if ep==hyperparams['decay_epoch']-1: scheduler.step(l)
    if (ep+1)%print_every==0: print(f'Epoch {ep+1}, Loss {l:.4f}')
print('Training time:', time.time()-start)
plot_losses(losses)

In [ ]:
# -----------------------------------------------------------------------------
# 9. Model Saving
# -----------------------------------------------------------------------------
ckpt={
    'state':model.state_dict(),
    'hyperparams':hyperparams,
    'stages':stages_cfg,
    'pooling':pool_cfg
}
torch.save(ckpt,'classifier_spec10_conv.pth')
print('Saved to classifier_spec10_conv.pth')